# 01 - Trace capture (fixed-key attack set + random-key profiling set)

## APPROVAL REQUIRED: DO NOT RUN UNATTENDED

This notebook **drives the target and claims USB** for bulk capture. It is approval-gated
for hardware safety and logged in `notes/approvals.md`. It requires the bring-up baseline
and a flashed `aes-unprotected` target.

Goal: capture a **fixed-key** set (~5k traces, random plaintexts) and a separate
**random-key** profiling set, save each to `traces/*.npz` + manifest via `dlsca.dataset`,
and sanity-plot alignment around the first-round S-box.

Datasets written: `traces/unprotected_fixedkey.npz` (CPA) and
`traces/unprotected_randomkey.npz` (profiling).

In [ ]:
# --- APPROVAL GUARD (placeholder) -------------------------------------------------
# Leave APPROVED = False in version control. Flip to True ONLY in a live, human-approved
# session after logging the approval in notes/approvals.md. Prevents unattended capture.
APPROVED = False
APPROVAL_REF = "notes/approvals.md#2026-05-31-capture-unprotected"
assert APPROVED is True, (
    "This hardware step is approval-gated. Get + log approval in notes/approvals.md, "
    "then set APPROVED = True. See notebook header."
)

In [ ]:
import os, pathlib, platform
import numpy as np
import matplotlib.pyplot as plt
import chipwhisperer as cw
from dlsca import seeds, capture, dataset

# Run from repo root so traces/ and results/ resolve regardless of kernel launch dir.
_p = pathlib.Path.cwd()
for _cand in (_p, *_p.parents):
    if (_cand / "src" / "dlsca").is_dir():
        os.chdir(_cand); break

SEED = 0
seeds.set_all(SEED)  # reproducible plaintext/key draws
N_FIXED = 5000      # attack set size
N_RANDOM = 5000     # profiling set size
FIXED_KEY = np.arange(16, dtype=np.uint8)  # 000102...0f (the verified key)

# Manifest fields recorded with every campaign (trace-dataset manifest schema).
base_manifest = {
    "firmware": "aes-unprotected",
    "firmware_hash": "sha256:ee9e2630f97037960f24a82b75968cacf623ae6b25395383228ab6d6dc55ec8a",
    "board": "ChipWhisperer-Nano (CWNANO)",
    "target": "STM32F0 (STM32F0_NANO, built-in)",
    "host_arch": platform.machine(),
    "scope": {"sample_rate": 7500000, "samples": 5000, "gain": None,
               "adc_clock": "7.5MHz", "trigger": "tio/simpleserial"},
    "cw_version": cw.__version__,
}

In [ ]:
# Connect once (claims USB, gated). Assumes aes-unprotected already flashed.
scope, target = capture.connect(approved=APPROVED, approval_ref=APPROVAL_REF)
print(scope)

In [ ]:
# Fixed-key attack campaign -> traces/unprotected_fixedkey.npz + manifest (CPA priority).
fixed = capture.campaign(scope, target, role="fixed-key", n=N_FIXED,
                         base_manifest=base_manifest, key_mode="fixed",
                         fixed_key=FIXED_KEY, seed=SEED,
                         approved=APPROVED, approval_ref=APPROVAL_REF)
dataset.save("unprotected_fixedkey", fixed.traces, fixed.plaintexts,
             fixed.keys, fixed.ciphertexts, fixed.manifest, overwrite=True)
print(dataset.validate(fixed, sample=500))

In [ ]:
# Random-key profiling campaign -> traces/unprotected_randomkey.npz + manifest.
rand = capture.campaign(scope, target, role="random-key", n=N_RANDOM,
                        base_manifest=base_manifest, key_mode="random", seed=SEED + 1,
                        approved=APPROVED, approval_ref=APPROVAL_REF)
dataset.save("unprotected_randomkey", rand.traces, rand.plaintexts,
             rand.keys, rand.ciphertexts, rand.manifest, overwrite=True)
print(dataset.validate(rand, sample=500))

In [ ]:
# Sanity: overlay a few traces to confirm trigger alignment around round 1.
ts = dataset.load("unprotected_fixedkey")
plt.figure(figsize=(9, 3))
for i in range(5):
    plt.plot(ts.traces[i], alpha=0.6, lw=0.5)
plt.title("Alignment check: first 5 fixed-key traces")
plt.xlabel("sample"); plt.ylabel("power (a.u.)")
plt.tight_layout(); plt.savefig("results/snr_unprotected.png", dpi=120); plt.show()

scope.dis()  # release the USB device when done